# Bank Marketing Dataset — Exploratory Data Analysis

This notebook completes all 5 requested tasks using **Pandas** and **Seaborn**.

**Dataset:** Kaggle `bank-additional-full.csv` (Portuguese bank telemarketing campaign)
**Target:** `y` — whether the customer subscribed to a term deposit (`yes` / `no`).

> The original dataset contains 41,188 records, 20 input variables, and the target variable `y`.


## 0. Setup

In [ ]:
# Install if needed (uncomment the next line in a fresh environment)
# %pip install pandas seaborn matplotlib kagglehub

from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Consistent visual defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


## 1. Download the Kaggle Bank Marketing Dataset, load it into `bank_df`, and inspect it

In [ ]:
# Download the public Kaggle dataset with KaggleHub.
# You may be prompted for Kaggle credentials only if your environment requires them.
import kagglehub

KAGGLE_DATASET = "sahistapatel96/bankadditionalfullcsv"
download_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))

# Find the CSV inside the downloaded dataset folder.
csv_candidates = list(download_dir.rglob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No CSV file found inside {download_dir}")

csv_path = csv_candidates[0]
print(f"Using file: {csv_path}")

# The Kaggle file is semicolon-separated.
bank_df = pd.read_csv(csv_path, sep=";")

print("First 10 rows:")
display(bank_df.head(10))

print("\nDataFrame info():")
bank_df.info()


## 2. Countplot — term-deposit subscriptions (`y`)

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=bank_df, x="y")
ax.set_title("Term Deposit Subscription Count")
ax.set_xlabel("Subscribed to Term Deposit (y)")
ax.set_ylabel("Number of Customers")

# Add counts on top of the bars.
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_subscription_countplot.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Boxplot — customer age vs. term-deposit subscription

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.boxplot(data=bank_df, x="y", y="age")
ax.set_title("Customer Age by Term Deposit Subscription")
ax.set_xlabel("Subscribed to Term Deposit (y)")
ax.set_ylabel("Customer Age (years)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_age_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Bar plot — subscription percentage by education level

In [ ]:
# Calculate the subscription percentage for each education level.
education_rate = (
    bank_df.assign(subscribed=(bank_df["y"] == "yes").astype(int))
    .groupby("education")["subscribed"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

education_plot_df = education_rate.reset_index(name="subscription_rate")

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=education_plot_df,
    x="education",
    y="subscription_rate"
)
ax.set_title("Term Deposit Subscription Rate by Education Level")
ax.set_xlabel("Education Level")
ax.set_ylabel("Customers Subscribed (%)")
ax.tick_params(axis="x", rotation=25)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_education_subscription_rate.png", dpi=150, bbox_inches="tight")
plt.show()

print("Subscription rate by education level:")
display(education_plot_df)


## 5. Two additional visualizations for Cred + implementation of one

### ChatGPT-suggested additional visualizations

1. **Job vs. subscription rate:** Compare the percentage of customers subscribing within each job category. This can help Cred identify occupations with stronger product-fit signals.
2. **Previous campaign outcome (`poutcome`) vs. subscription rate:** Compare conversion rates for customers whose earlier campaign was a success, failure, or had no prior outcome. This can help Cred prioritize customers with strong prior engagement.

Below, the **job vs. subscription rate** visualization is implemented using Seaborn.


In [ ]:
# Additional visualization: subscription rate by job category.
job_rate = (
    bank_df.assign(subscribed=(bank_df["y"] == "yes").astype(int))
    .groupby("job")["subscribed"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

job_plot_df = job_rate.reset_index(name="subscription_rate")

plt.figure(figsize=(11, 6))
ax = sns.barplot(
    data=job_plot_df,
    x="subscription_rate",
    y="job"
)
ax.set_title("Term Deposit Subscription Rate by Job")
ax.set_xlabel("Customers Subscribed (%)")
ax.set_ylabel("Job")

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_job_subscription_rate.png", dpi=150, bbox_inches="tight")
plt.show()

print("Subscription rate by job:")
display(job_plot_df)


## 6. Quick business takeaways for Cred

In [ ]:
overall_rate = (bank_df["y"] == "yes").mean() * 100
best_education = education_rate.index[0]
best_education_rate = education_rate.iloc[0]
best_job = job_rate.index[0]
best_job_rate = job_rate.iloc[0]

print(f"Overall term-deposit subscription rate: {overall_rate:.2f}%")
print(f"Highest education-level subscription rate: {best_education} ({best_education_rate:.2f}%)")
print(f"Highest job-category subscription rate: {best_job} ({best_job_rate:.2f}%)")

print("\nBusiness interpretation:")
print("Use these rates as descriptive signals for customer targeting—not as causal evidence.")
print("For a production savings-product model, combine demographic, behavioral, product-use, and engagement signals and validate on an out-of-sample holdout set.")
